In [1]:
import pandas as pd
from pathlib import Path
from shapely import wkb

In [2]:
# Define the workspace directory containing the parquet files
WORKSPACE_DIR = Path("/Users/muhammadabdul/Desktop/penn/earth_genome_internship/climate_trace/data/KS_barton_lot_pond")

In [3]:
# Get all parquet files in the workspace directory
parquet_files = sorted(WORKSPACE_DIR.glob("*sp.parquet"))

# Check if any parquet files were found
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in: {WORKSPACE_DIR}")

In [4]:
# Initialize empty DataFrames for each facility type
lot_merged = pd.DataFrame()
pond_merged = pd.DataFrame()
other_merged = pd.DataFrame()

In [5]:
# Iterate through each parquet file, read it into a DataFrame, and merge based on facility type
for parquet_file in parquet_files:
    tmp_df = pd.read_parquet(parquet_file )

    # Add a new column 'facility_type' based on the file name
    if 'other' in parquet_file.name:
        tmp_df['facility_type'] = 'other'
    elif 'pond' in parquet_file.name:
        tmp_df['facility_type'] = 'pond'
    elif 'lot' in parquet_file.name:
        tmp_df['facility_type'] = 'lot'

    # Merge the DataFrame into the corresponding merged DataFrame based on facility type while avoiding duplicate columns
    if 'lot' in parquet_file.name:
        lot_merged = pd.concat([lot_merged, tmp_df], axis = 1)
        lot_merged = lot_merged.loc[:,~lot_merged.columns.duplicated()]
    elif 'pond' in parquet_file.name:
        pond_merged = pd.concat([pond_merged, tmp_df], axis = 1)
        pond_merged = pond_merged.loc[:,~pond_merged.columns.duplicated()]
    elif 'other' in parquet_file.name:
        other_merged = pd.concat([other_merged, tmp_df], axis = 1)
        other_merged = other_merged.loc[:,~other_merged.columns.duplicated()]

In [6]:
# Convert the 'geometry' column from WKB to Shapely geometry objects if they are in bytes format
lot_merged["geometry"] = lot_merged["geometry"].apply(lambda g: wkb.loads(g) if isinstance(g, (bytes, bytearray)) else g)
pond_merged["geometry"] = pond_merged["geometry"].apply(lambda g: wkb.loads(g) if isinstance(g, (bytes, bytearray)) else g)
other_merged["geometry"] = other_merged["geometry"].apply(lambda g: wkb.loads(g) if isinstance(g, (bytes, bytearray)) else g)

In [ ]:
# Concatenate the merged DataFrames for each facility type into a single DataFrame and save
df_merged = pd.concat([lot_merged, pond_merged, other_merged], axis = 0)
df_merged.to_csv(WORKSPACE_DIR / "merged_data_embeddings_sp.csv", index=False)